# 店舗売上需要予測: 実験 #002 ドメイン特徴量拡張モデル

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** カレンダー拡張 ＋ 給与日（15日・月末） ＋ 祝日振替 ＋ 店舗メタ情報 ＋ 原油価格の統合による精度向上
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **モデル:** **LightGBM Regressor**
- **前スコア:** 実験 #001 ベースライン: **LB 0.52338**

---

## 1. 今回追加するドメイン知識特徴量
1. **エクアドル給与日フラグ (`is_wage_day` & `is_wage_day_after`):**
   - 公務員の給与支給日である「毎月15日」と「月末」、およびその翌日（買い出しピーク）のフラグ。
2. **店舗メタ情報 (`stores.csv`):**
   - 店舗の都市 (`city`)、州 (`state`)、店舗タイプ (`store_type`)、クラスタ (`cluster`) の属性を付与。
3. **祝日振替・イベント情報 (`holidays_events.csv`):**
   - 振替日 (`transferred == True`) を除外し、真の公休日・全国祝日をフラグ化。
4. **原油価格情報 (`oil.csv`):**
   - 原油価格の線形補間値と7日間移動平均 (`oil_7d_mean`) を結合（エクアドル経済指標）。
5. **震災イベントフラグ (`is_earthquake`):**
   - 2016年4月16日の大地震 (M7.8) 後の特需期間をフラグ化。

## 2. ライブラリ読み込み & データ読み込み

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# データディレクトリ自動検出
def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '/kaggle/input/store-sales-time-series-forecasting',
        '003_store-sales/data',
        '../data',
        'data'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('**/store-sales-time-series-forecasting/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

print(f'訓練データ形状: {train_df.shape}')
print(f'テストデータ形状: {test_df.shape}')

## 3. ドメイン特徴量エンジニアリング（5テーブルの統合）

In [ ]:
# 1. カレンダー & 給与日特徴量
for df in [train_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['dayofyear'] = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    # 給与日（15日と月末）
    df['is_wage_day'] = ((df['day'] == 15) | (df['is_month_end'] == 1)).astype(int)
    df['is_wage_day_after'] = ((df['day'] == 16) | (df['is_month_start'] == 1)).astype(int)
    # 2016年大地震フラグ
    df['is_earthquake'] = ((df['date'] >= '2016-04-16') & (df['date'] <= '2016-05-16')).astype(int)

# 2. 店舗メタ情報 (stores.csv)
stores_path = os.path.join(data_dir, 'stores.csv')
if os.path.exists(stores_path):
    stores_df = pd.read_csv(stores_path).rename(columns={'type': 'store_type'})
    train_df = train_df.merge(stores_df, on='store_nbr', how='left')
    test_df = test_df.merge(stores_df, on='store_nbr', how='left')

# 3. 祝日振替・イベント情報 (holidays_events.csv)
holidays_path = os.path.join(data_dir, 'holidays_events.csv')
if os.path.exists(holidays_path):
    holidays_df = pd.read_csv(holidays_path)
    holidays_df['date'] = pd.to_datetime(holidays_df['date'])
    # 振替除外、平日出勤除外
    real_holidays = holidays_df[~holidays_df['transferred']]
    real_holidays = real_holidays[real_holidays['type'] != 'Work Day']
    nat_holidays = real_holidays[real_holidays['locale'] == 'National'][['date']].drop_duplicates()
    nat_holidays['is_national_holiday'] = 1
    train_df = train_df.merge(nat_holidays, on='date', how='left')
    test_df = test_df.merge(nat_holidays, on='date', how='left')
    train_df['is_national_holiday'] = train_df['is_national_holiday'].fillna(0).astype(int)
    test_df['is_national_holiday'] = test_df['is_national_holiday'].fillna(0).astype(int)

# 4. 原油価格情報 (oil.csv)
oil_path = os.path.join(data_dir, 'oil.csv')
if os.path.exists(oil_path):
    oil_df = pd.read_csv(oil_path)
    oil_df['date'] = pd.to_datetime(oil_df['date'])
    all_dates = pd.date_range(start=oil_df['date'].min(), end='2017-08-31')
    oil_full = pd.DataFrame({'date': all_dates}).merge(oil_df, on='date', how='left')
    oil_full['dcoilwtico'] = oil_full['dcoilwtico'].interpolate(method='linear').bfill()
    oil_full['oil_7d_mean'] = oil_full['dcoilwtico'].rolling(7, min_periods=1).mean()
    train_df = train_df.merge(oil_full[['date', 'dcoilwtico', 'oil_7d_mean']], on='date', how='left')
    test_df = test_df.merge(oil_full[['date', 'dcoilwtico', 'oil_7d_mean']], on='date', how='left')

# カテゴリ変数の指定
cat_cols = ['store_nbr', 'family', 'city', 'state', 'store_type', 'cluster']
for col in cat_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype('category')
        test_df[col] = test_df[col].astype('category')

ignore_cols = ['id', 'date', 'sales']
features = [c for c in train_df.columns if c not in ignore_cols]
print(f'総特徴量数: {len(features)} 列')
print(f'特徴量: {features}')

## 4. 時系列バリデーション分割 (Time-based Split)

In [ ]:
val_start = '2017-07-31'
val_end = '2017-08-15'
train_start = '2016-01-01'

trn_mask = (train_df['date'] >= train_start) & (train_df['date'] < val_start)
val_mask = (train_df['date'] >= val_start) & (train_df['date'] <= val_end)

X_train, y_train = train_df.loc[trn_mask, features], train_df.loc[trn_mask, 'sales'].values
X_val, y_val = train_df.loc[val_mask, features], train_df.loc[val_mask, 'sales'].values
X_test = test_df[features]

print(f'学習サンプル数: {len(X_train):,} 件')
print(f'検証サンプル数: {len(X_val):,} 件')

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

## 5. LightGBM 回帰モデルの学習

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'random_state': 42,
    'verbose': -1,
    'n_estimators': 1000,
}

model = lgb.LGBMRegressor(**params)
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100)
]

model.fit(
    X_train,
    y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=callbacks,
)

# 特徴量重要度 Top 10 の可視化
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
importances.tail(15).plot(kind='barh', color='#1f77b4')
plt.title('LightGBM Feature Importance (Top 15)')
plt.xlabel('Importance (Split)')
plt.tight_layout()
plt.show()

## 6. 手元検証スコア（RMSLE）の評価

In [ ]:
val_pred_log = model.predict(X_val)
val_pred = np.clip(np.expm1(val_pred_log), 0, None)

rmsle = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(val_pred)))
print('=' * 55)
print(f'  [実験 #002] 手元検証スコア (Val RMSLE): {rmsle:.5f}')
print('=' * 55)

# 実測値 vs 予測値の推移プロット
val_df = train_df.loc[val_mask].copy()
val_df['pred_sales'] = val_pred
daily_val = val_df.groupby('date')[['sales', 'pred_sales']].sum().reset_index()

plt.figure(figsize=(12, 4))
plt.plot(daily_val['date'], daily_val['sales'], label='Actual (y_true)', marker='o', color='#1f77b4')
plt.plot(daily_val['date'], daily_val['pred_sales'], label='Predicted (y_pred)', marker='x', linestyle='--', color='#ff7f0e')
plt.title(f'Validation: Actual vs Predicted (RMSLE: {rmsle:.5f})')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 7. テストデータ予測 & 提出ファイル作成

In [ ]:
test_pred_log = model.predict(X_test)
test_pred = np.clip(np.expm1(test_pred_log), 0, None)

sub_df['sales'] = test_pred
sub_df.to_csv('submission.csv', index=False)

print('提出用 submission.csv を作成しました！')
print(f'行数: {len(sub_df):,} 行')
print(sub_df.head(10))
print(sub_df.describe())